# V20 – Datenbanken Teil 2: Praxis

## 🎯 Lernziele
Nach diesem Notebook kannst du …
- eine kleine deterministische In-Memory-DB mit `setup_db()` aufbauen,
- `INNER JOIN`, `LEFT JOIN`, `GROUP BY`, `HAVING` und eine einfache Subquery praktisch anwenden,
- erkennen, wann `INNER JOIN` und wann `LEFT JOIN` das richtige Werkzeug ist.

## ⏱️ Zeitbudget
Ca. 25 Minuten.

## 🧭 TL;DR
Wir legen eine kleine, feste Maschinen- und Wartungs-Datenbank an und üben die Theorie Schritt für Schritt an konkreten Abfragen. Drei Fill-in-Blanks bauen dir unterwegs Muskelgedächtnis auf.

## 📦 Voraussetzungen
- [01_theorie.ipynb](01_theorie.ipynb).


## 1. `setup_db()` – unsere feste Datenbasis

Damit alle Ergebnisse reproduzierbar bleiben, legen wir die gesamte Datenbasis in einer einzigen Funktion `setup_db()` an. Sie öffnet eine frische In-Memory-DB, legt die Tabellen `Maschinen` und `Wartungen` an und füllt sie mit **denselben** festen Zeilen – ohne `random`, ohne externe Dateien. Jede Zelle, die mit `conn = setup_db()` beginnt, startet dadurch aus exakt demselben Zustand.

In [1]:
import sqlite3

def setup_db() -> sqlite3.Connection:
    """Legt eine frische In-Memory-DB mit deterministischen Daten an."""
    conn = sqlite3.connect(":memory:")
    cur = conn.cursor()
    cur.execute("""
        CREATE TABLE Maschinen (
            Maschinen_ID INTEGER PRIMARY KEY,
            Name         TEXT NOT NULL,
            Typ          TEXT NOT NULL,
            Baujahr      INTEGER NOT NULL
        )
    """)
    cur.execute("""
        CREATE TABLE Wartungen (
            Wartung_ID    INTEGER PRIMARY KEY,
            Maschinen_ID  INTEGER NOT NULL,
            Wartungstyp   TEXT NOT NULL,
            Datum         TEXT NOT NULL,
            Kosten        REAL NOT NULL,
            FOREIGN KEY (Maschinen_ID) REFERENCES Maschinen(Maschinen_ID)
        )
    """)
    maschinen = [
        (1, "CNC-Fräse 01",    "Fräse",    2018),
        (2, "CNC-Fräse 02",    "Fräse",    2020),
        (3, "Drehmaschine 01", "Drehbank", 2016),
        (4, "Drehmaschine 02", "Drehbank", 2021),
        (5, "Roboter-Arm R1",  "Roboter",  2019),
    ]
    wartungen = [
        (1, 1, "Inspektion",   "2024-01-10",  150.00),
        (2, 1, "Reparatur",    "2024-03-05", 1200.00),
        (3, 2, "Inspektion",   "2024-02-14",  180.00),
        (4, 3, "Ölwechsel",    "2024-01-20",   90.00),
        (5, 3, "Reparatur",    "2024-04-12", 2500.00),
        (6, 3, "Inspektion",   "2024-05-18",  200.00),
        (7, 4, "Inspektion",   "2024-02-28",  160.00),
        (8, 5, "Kalibrierung", "2024-03-22",  450.00),
    ]
    cur.executemany("INSERT INTO Maschinen VALUES (?, ?, ?, ?)", maschinen)
    cur.executemany("INSERT INTO Wartungen VALUES (?, ?, ?, ?, ?)", wartungen)
    conn.commit()
    return conn

# Kurztest
conn = setup_db()
cur = conn.cursor()
cur.execute("SELECT COUNT(*) FROM Maschinen")
print("Maschinen:", cur.fetchone()[0])
cur.execute("SELECT COUNT(*) FROM Wartungen")
print("Wartungen:", cur.fetchone()[0])
conn.close()


Maschinen: 5
Wartungen: 8


## 2. `INNER JOIN` – Wartung + Maschine zusammenführen

Wir fragen zu jeder Wartung den passenden Maschinennamen ab. Das klassische JOIN-Muster: `FROM Maschinen M INNER JOIN Wartungen W ON M.Maschinen_ID = W.Maschinen_ID`.

In [2]:
conn = setup_db()
cur = conn.cursor()
cur.execute("""
    SELECT M.Name, W.Wartungstyp, W.Kosten
    FROM Maschinen M
    INNER JOIN Wartungen W ON M.Maschinen_ID = W.Maschinen_ID
    ORDER BY M.Name, W.Datum
""")
for zeile in cur.fetchall():
    print(zeile)
conn.close()


('CNC-Fräse 01', 'Inspektion', 150.0)
('CNC-Fräse 01', 'Reparatur', 1200.0)
('CNC-Fräse 02', 'Inspektion', 180.0)
('Drehmaschine 01', 'Ölwechsel', 90.0)
('Drehmaschine 01', 'Reparatur', 2500.0)
('Drehmaschine 01', 'Inspektion', 200.0)
('Drehmaschine 02', 'Inspektion', 160.0)
('Roboter-Arm R1', 'Kalibrierung', 450.0)


## 3. `LEFT JOIN` – auch Maschinen ohne Wartungen anzeigen

Zum Vergleich fügen wir testweise eine sechste Maschine `Roboter-Arm R2` ein, die noch **keine** Wartung hatte. Mit `INNER JOIN` würde sie unsichtbar bleiben; mit `LEFT JOIN` erscheint sie mit `NULL`-Werten auf der rechten Seite.

In [3]:
conn = setup_db()
cur = conn.cursor()
cur.execute("INSERT INTO Maschinen VALUES (?, ?, ?, ?)",
            (6, "Roboter-Arm R2", "Roboter", 2023))

cur.execute("""
    SELECT M.Name, W.Wartungstyp, W.Kosten
    FROM Maschinen M
    LEFT JOIN Wartungen W ON M.Maschinen_ID = W.Maschinen_ID
    ORDER BY M.Name
""")
for zeile in cur.fetchall():
    print(zeile)
conn.close()


('CNC-Fräse 01', 'Inspektion', 150.0)
('CNC-Fräse 01', 'Reparatur', 1200.0)
('CNC-Fräse 02', 'Inspektion', 180.0)
('Drehmaschine 01', 'Inspektion', 200.0)
('Drehmaschine 01', 'Reparatur', 2500.0)
('Drehmaschine 01', 'Ölwechsel', 90.0)
('Drehmaschine 02', 'Inspektion', 160.0)
('Roboter-Arm R1', 'Kalibrierung', 450.0)
('Roboter-Arm R2', None, None)


Beachte die letzte Zeile: `("Roboter-Arm R2", None, None)`. Genau das sucht man in Aufgabe 5 heraus.

## 4. `GROUP BY` + `SUM` – Top-Kostenverursacher

Wer hat uns am meisten gekostet? Wir gruppieren die Wartungen pro Maschine und addieren die Kosten. `ORDER BY gesamt DESC LIMIT 1` liefert den Spitzenreiter.

In [4]:
conn = setup_db()
cur = conn.cursor()
cur.execute("""
    SELECT M.Name, SUM(W.Kosten) AS gesamt
    FROM Maschinen M
    INNER JOIN Wartungen W ON M.Maschinen_ID = W.Maschinen_ID
    GROUP BY M.Maschinen_ID
    ORDER BY gesamt DESC
""")
for zeile in cur.fetchall():
    print(f"{zeile[0]:20s} {zeile[1]:>8.2f} €")
conn.close()


Drehmaschine 01       2790.00 €
CNC-Fräse 01          1350.00 €
Roboter-Arm R1         450.00 €
CNC-Fräse 02           180.00 €
Drehmaschine 02        160.00 €


## 5. `GROUP BY` + `COUNT` – Wartungstypen zählen

Welche Art Wartung kommt am häufigsten vor? Wir gruppieren direkt nach `Wartungstyp`.

In [5]:
conn = setup_db()
cur = conn.cursor()
cur.execute("""
    SELECT Wartungstyp, COUNT(*) AS anzahl
    FROM Wartungen
    GROUP BY Wartungstyp
    ORDER BY anzahl DESC, Wartungstyp
""")
print(cur.fetchall())
conn.close()


[('Inspektion', 4), ('Reparatur', 2), ('Kalibrierung', 1), ('Ölwechsel', 1)]


## 6. `HAVING` – nur teure Maschinen

Wir wollen jetzt nur die Maschinen sehen, deren **summierte** Wartungskosten 1000 € übersteigen. Das ist ein Filter **auf Gruppenebene** und muss deshalb mit `HAVING` formuliert werden:

In [6]:
conn = setup_db()
cur = conn.cursor()
cur.execute("""
    SELECT M.Name, SUM(W.Kosten) AS gesamt
    FROM Maschinen M
    INNER JOIN Wartungen W ON M.Maschinen_ID = W.Maschinen_ID
    GROUP BY M.Maschinen_ID
    HAVING gesamt > 1000
    ORDER BY gesamt DESC
""")
print(cur.fetchall())
conn.close()


[('Drehmaschine 01', 2790.0), ('CNC-Fräse 01', 1350.0)]


## 7. Subquery – über dem Durchschnitt

Mit einer skalaren Subquery können wir nach Maschinen fragen, deren Gesamtkosten **über dem Durchschnitt** aller Gesamtkosten liegen.

In [7]:
conn = setup_db()
cur = conn.cursor()
cur.execute("""
    SELECT Name FROM Maschinen
    WHERE Maschinen_ID IN (
        SELECT Maschinen_ID FROM Wartungen
        GROUP BY Maschinen_ID
        HAVING SUM(Kosten) > (SELECT AVG(Kosten) FROM Wartungen)
    )
""")
print(cur.fetchall())
conn.close()


[('CNC-Fräse 01',), ('Drehmaschine 01',)]


## 8. Fill-in-Blank 1 – `COUNT`

Ergänze die Abfrage so, dass `anzahl` auf die Gesamtzahl der Einträge in `Wartungen` gesetzt wird (erwarteter Wert: 8).

In [8]:
conn = setup_db()
cur = conn.cursor()

# TODO: führe die passende SQL-Abfrage aus
cur.execute("SELECT 0")  # ersetzen!
anzahl = cur.fetchone()[0]
conn.close()

try:
    assert anzahl == 8, f"Erwartet 8, bekommen: {anzahl}"
    print("✅ Fill-in 1 gelöst.")
except AssertionError as e:
    print(f"❌ Noch nicht ganz: {e}")
except NameError:
    print("❌ Variable `anzahl` fehlt.")


❌ Noch nicht ganz: Erwartet 8, bekommen: 0


## 9. Fill-in-Blank 2 – `SUM` + `GROUP BY`

Fülle die Lücken so, dass `ergebnis` eine Liste `[(Wartungstyp, Summe der Kosten)]` enthält, sortiert **alphabetisch** nach Wartungstyp.

In [9]:
conn = setup_db()
cur = conn.cursor()

# TODO: passende GROUP-BY-Abfrage
cur.execute("SELECT 'TODO', 0 WHERE 0")  # ersetzen!
ergebnis = cur.fetchall()
conn.close()

try:
    assert ergebnis == [
        ("Inspektion",    690.0),
        ("Kalibrierung",  450.0),
        ("Reparatur",    3700.0),
        ("Ölwechsel",      90.0),
    ], f"Ergebnis stimmt noch nicht: {ergebnis}"
    print("✅ Fill-in 2 gelöst.")
except AssertionError as e:
    print(f"❌ Noch nicht ganz: {e}")
except NameError:
    print("❌ Variable `ergebnis` fehlt.")


❌ Noch nicht ganz: Ergebnis stimmt noch nicht: []


## 10. Fill-in-Blank 3 – `INNER JOIN` + `MAX`

Schreibe eine Abfrage, die den Namen der Maschine und die Kosten der **teuersten einzelnen Wartung** zurückliefert. Das Ergebnis soll in `teuerste` stehen und ein Tupel `(Name, Kosten)` sein.

In [10]:
conn = setup_db()
cur = conn.cursor()

# TODO: JOIN + ORDER BY Kosten DESC LIMIT 1
cur.execute("SELECT 'TODO', 0.0")  # ersetzen!
teuerste = cur.fetchone()
conn.close()

try:
    assert teuerste == ("Drehmaschine 01", 2500.0), f"Erwartet ('Drehmaschine 01', 2500.0), bekommen: {teuerste}"
    print("✅ Fill-in 3 gelöst.")
except AssertionError as e:
    print(f"❌ Noch nicht ganz: {e}")
except NameError:
    print("❌ Variable `teuerste` fehlt.")


❌ Noch nicht ganz: Erwartet ('Drehmaschine 01', 2500.0), bekommen: ('TODO', 0.0)


## ✅ Zusammenfassung
- `setup_db()` liefert uns eine reproduzierbare Ausgangslage.
- `INNER JOIN` und `LEFT JOIN` unterscheiden sich nur darin, ob Zeilen ohne Partner wegfallen.
- `GROUP BY` + Aggregatfunktion + `ORDER BY` ist das Standard-Muster für Kennzahlen.
- `HAVING` filtert **Gruppen**, Subqueries erlauben Zwischenergebnisse als Bausteine.

## ➡️ Nächster Schritt
Weiter mit [03_aufgaben.ipynb](03_aufgaben.ipynb).
